# Monte Carlo grain structure generation in 3D

This notebook grows a 3D grain structure with UPXO's Monte Carlo (Potts) grain growth model, then pokes at the result: grain morphology, intercept grain sizes, grain queries, sub-domains and a quick visualisation. At the end it saves the grain labels as a `.npy` file you can use for meshing.

You need `demo3D1.xls` (the input dashboard with the simulation settings) in the same folder as this notebook. Start Jupyter from that folder so the notebook finds it.

In [ ]:
from upxo.ggrowth.mcgs import mcgs
from scipy.spatial import cKDTree
import numpy as np
import pyvista as pv
from copy import deepcopy
import matplotlib.pyplot as plt
import upxo._sup.data_ops as DO
import upxo._sup.dataTypeHandlers as dth
from pathlib import Path

## Input dashboard

The grid and simulation settings live in an Excel dashboard. `DASHBOARD_FILEPATH` points at it; by default that's `demo3D1.xls` sitting next to this notebook. To use your own dashboard, give its full path instead.

In [ ]:
# The dashboard sits next to this notebook. For your own file, give its full path instead if needed.
# The full path version of the dashboard, if needed needs to be entered as below:
# input_dashboard='C:\\Development\\UPXO\\upxo_demos\\gsgen\\demo2D1.xls'

DASHBOARD_FILEPATH = Path.cwd() / 'demo3D1.xls'

## Create the Monte Carlo object

This reads the dashboard and builds the `mcgs` object, `pxt`. Nothing has been simulated yet.

In [ ]:
"""Instantiate ythe mcgs class"""
pxt = mcgs(study='independent', input_dashboard=DASHBOARD_FILEPATH)

Let's look at the data structure a bit.

In [ ]:
pxt

In [ ]:
pxt.uigrid

In [ ]:
pxt.uisim

In [ ]:
pxt.vox_size

In [ ]:
pxt.vox_length

Simulate the grain structuer and inspect the data-structure.

In [ ]:
pxt.simulate(verbose=False)

Sinulation has created temporally evolved grain structure. These are
contained in pxt.gs. We will now inspect this and provide details of it below.

In [ ]:
print(pxt.gs.keys(), '\n')
print(f"First temporal slice is ---> {pxt.gs[0]}\n", 20*'-  ')

This is a dictionatry containing monte-carlo time step as keys and the corresponding UPXO grain structue objects as the value.

Thoushgh grain struvcture have been created, these are not fully populated with the information needed for the completion of either conformal meshing pipeline or the representativeness qualification pipeline or any other major thing for that matter. We will achivve these in steps and at the complettion of each step, the necessary data will be populated and enable user to perform certain functyions / operations. We will not, at this moment look at all details of the this object, but rather look at its details in steps below.

Lets extract a slice (say 5th) from the grain structue temporal stack

In [ ]:
tslice = pxt.m[-1]
gstslice = pxt.gs[tslice]
print(gstslice)

In [ ]:
gstslice.char_morphology_of_grains(label_str_order=3,
                                   find_grain_voxel_locs=True,
                                   find_spatial_bounds_of_grains=True,
                                   force_compute=True)
gstslice.set_mprops(volnv=True, eqdia=True, eqdia_base_size_spec='volnv',
                    arbbox=False, arbbox_fmt='gid_dict',
                    arellfit=False, arellfit_metric='max', arellfit_calculate_efits=True,
                    arellfit_efit_routine=1, arellfit_efit_regularize_data=True,
                    solidity=False, sol_nan_treatment='replace', sol_inf_treatment='replace',
                    sol_nan_replacement=-1, sol_inf_replacement=-1,
                    sanv=False, sanv_N=26, rat_sanv_volnv=False, sanv_verbosity=1E2, disp_msg='')

All moorphologuical properti3es can be accessed as below.

In [ ]:
print(gstslice.mprop.keys())

The keys and thier ecxplanations as of 15-08-2024 is proided below:
- volnv: Volume by number of voxels
- volsr: Volume after grain boundary surface reconstruction
- volch: Volume of convex hull
- sanv: surface area by number of voxels
- savi: surface area by voxel interfaces
- sasr: surface area after grain boundary surface reconstruction
- pernv: perimeter by number of voxels
- pervl: perimeter by voxel edge lines
- pergl: perimeter by geometric grain boundary line segments
- eqdia: eqvivalent diameter
- kx: grain boundary voxel local curvature in yz plane
- ky: grain boundary voxel local curvature in xz plane
- kz: grain boundary voxel local curvature in xy plane
- kxyz: mean(kx, ky, kz)
- ksr: k computed from surface reconstruction.
- arbbox: aspect ratio by bounding box
- arefit: aspect ratio by ellipsoidal fit
- sol: solidity
- ecc: eccentricity - how much the shape of the grain differs from a sphere.
- com: compactness
- sph: sphericity
- fn: flatness
- rnd: roundness
- mi: moment of inertia tensor
- fdim: fractal dimension

We can use the following to extract specific grain structure IDs.

In [ ]:
gstslice.get_largest_gids()

In [ ]:
len(gstslice.get_smallest_gids())

In [ ]:
len(gstslice.single_voxel_grains)

In [ ]:
len(gstslice.small_grains(vth=2))  # vth: Volume Threshold

In [ ]:
len(gstslice.large_grains(vth=2))

In [ ]:
len(gstslice.find_grains_by_nvoxels(nvoxels=2))

The following properties can be used to querry specific morphologuical properties.

In [ ]:
low = gstslice.smallest_volume
high = gstslice.largest_volume
prop_range = high-low

gstslice.find_grains_by_mprop_range(prop_name='volnv', 
                                    low=low+prop_range*0.25,
                                    high=high+prop_range*0.75,
                                    low_ineq='ge',
                                    high_ineq='le')

You can use gstslice.gpos to know the relative locations of grains in the grain structure position. It is a nested dictionary

In [ ]:
print(gstslice.gpos)

We see that this is inexistent. This is because, it was not caculated using characterisation. We will do it now

In [ ]:
# gstslice.set_grain_positions(verbose=False)

In [ ]:
# gstslice.gpos.keys()

In [ ]:
# print(gstslice.gpos['face']['left'])

Let us extract values of a scalar variable along a stright line, with the
straight line being specified by starting and ending coordinate indice
locatoins.

This returns a numpy array of lgi values. The line between the two
specified end points is generated using Bresenham algorithm in 3D.

In [ ]:
gstslice.get_values_along_line([0, 0, 0], [9, 9, 9], scalars='lgi')

You can also calculate the intercept - grain size in a couple of different
ways.

In [ ]:
gstslice.get_igs_properties_along_line([0, 0, 0], [9, 9, 9], scalars='lgi')

This gives a dictionary with the following keys:
* ng: Number of grains
* nv: np array of number ofvoxels between all grain boundaris on the line
* igs: intercept grain size (mean)
* igs_median: median value of the igs
* igs_range: range of grain sizes along the line
* igs_std: standard deviation of the of grain sizes along the line
* igs_var: variance of the of grain sizes along the line
* sv: scalar values along the line between the two specified locaytions
* sv_unique: unique scalar values along the line between the two specified locaytions

In [ ]:
gstslice.get_igs_along_line([0, 0, 0], [9, 9, 9], metric='mean', minimum=True,
                       maximum=True, std=True, variance=True, verbose=True)

This gives us a dictionaryu of following keys:
- 'igs': value of the metric of igs values.
- 'metric': metric specified by the user.
- 'min': Minimum value. Key only present if minimum is specified True.
- 'max': Maximum value. Key only present if maximum is specified True.
- 'std': Std. deviation value. Key only present if std is specified True.
- 'var': Variance value. Key only present if variance is specified True.

In [ ]:
gstslice.get_igs_along_lines(metric='mean', minimum=True, maximum=True,
                             std=True, variance=True, lines_gen_method=1,
                             lines_kwargs1={'plane': 'z',
                                            'start_skip1': 0,
                                            'start_skip2': 0,
                                            'incr1': 10, 'incr2': 10,
                                            'inclination': 'none',
                                            'inclination_extent': 0,
                                            'shift_seperately': False,
                                            'shift_starts': False,
                                            'shift_ends': False,
                                            'start_shift': 0, 'end_shift': 0})

This produces a dictionary with the folloiwng keys
* 'igs': overall intercept grain size of the grain structuer.
* 'metric': metric used for calulation of the overall igs value.
* 'min': minimum values of igs in each sampling line.
* 'max': maximum values of igs in each sampling line.
* 'std': standard deviation of igs values in each sampling line.
* 'var': varianbce of igs values in each sampline line.
* 'igs_all': individsual igs over each sampling line.
* 'ngrains': total number of grains in grain structure.

In [ ]:
gstslice.igs_sed_ratio(metric='mean', lines_gen_method=1,
                       reset_grain_size=True, base_size_spec='volnv',
                       lines_kwargs1={'plane': 'z',
                                      'start_skip1': 0, 'start_skip2': 0,
                                      'incr1': 3, 'incr2': 3,
                                      'inclination': 'random',
                                      'inclination_extent': 0,
                                      'shift_seperately': False,
                                      'shift_starts': False,
                                      'shift_ends': False,
                                      'start_shift': 0, 'end_shift': 0})

## Random sub-domains

The next few cells pull `n` random `p x q x r` blocks out of the grain structure (`feature_name='base'` means they come from `gstslice.lgi`) and can build PyVista grids of them for plotting. Heads up: these cells are wrapped in quotes, so they just show the code as text and don't run. Take the quotes off to try them.

In [ ]:
"A = gstslice.extract_subdomains_random(p=5, q=5, r=5, n=2, feature_name='base', user_fid=None, make_pvgrids=True)"

In [ ]:
"A.keys()"

In [ ]:
"len(A['sd'])"

In [ ]:
"A['pvgrids'][0]"

In [ ]:
"A['pvgrids'][0].plot()"

## Grain count and cleaning

`gstslice.n` is the number of grains. The cell after it shows how to merge small grains with `clean_gs_GMD_by_source_erosion_v1`: grains below `threshold` for the chosen property (`volnv`, the volume in voxels) get dissolved into their neighbours. That cell is also wrapped in quotes, so it doesn't run as it stands. Take the quotes off and the second `gstslice.n` will show how many grains the cleaning removed.

In [ ]:
gstslice.n

In [ ]:
"""gstslice.clean_gs_GMD_by_source_erosion_v1(prop='volnv', threshold=40,
                                           parameter_metric='mean',
                                           reset_pvgrid_every_iter=False,
                                           find_neigh_every_iter=False,
                                           find_grvox_every_iter=False,
                                           find_grspabnds_every_iter=False)"""

In [ ]:
gstslice.n

## Quick look

Shows the grain structure on three orthogonal slice planes, coloured by grain label (`lgi`). I have commented it out, so thast if you have jupyter notebook issues with pyvista viz, you can still go till the end of this notebook. The cell below is commented out; remove the `#` to run it.

In [ ]:
# gstslice.viz_mesh_slice_ortho(scalar='lgi', cmap='viridis', style='surface', throw=False, pvp=None)

## Relabel the grains

`cc3d.connected_components` relabels the grains using face connectivity, so every connected piece gets its own label. The result is `lfi`.

In [ ]:
import cc3d
lfi = cc3d.connected_components(gstslice.lfi, connectivity=6)
# cc3d by default uses 26-connectivity for 3D, but we specified 6-connectivity above

## View the relabelled grains

This plots `lfi` in 3D with PyVista, using `make_pvgrid` to turn the array into a grid and `plot_pvgrid` to show it. `scalar_name` is just the name given to the data on the grid; call it whatever you like. The cell is wrapped in quotes, so it doesn't run as it stands. Take the quotes off to try it.

In [ ]:
"""from upxo.viz import gsviz

gsviz.plot_pvgrid(gsviz.make_pvgrid(lfi, scalar_name='my_day'),
                  scalar_name='my_day', cmap='nipy_spectral', show_edges=True)"""

## Save for meshing

Saves the relabelled array as `UPXO_mcgs3d_01.npy` in the folder you started Jupyter from, which is the notebook's own folder by default. To save it somewhere else, put the full path in the `np.save` call.

In [ ]:
np.save(Path.cwd() / 'UPXO_mcgs3d_01.npy', lfi)

## Shuffle the grain IDs (optional)

`cc3d` labels grains in scan order, so neighbouring grains tend to get similar IDs, which makes colour maps look streaky. `shuffle_feature_IDs` randomises the IDs without touching the grain layout. Heads up: this cell sits after the save above, so the `.npy` file that was saved still has the unshuffled labels. Move this cell above the save if you want the shuffled ones saved.

In [ ]:
# Optionally, shuffle the feature IDs to randomize the spatially sequential labeling of gids by cc3d
import upxo.gsdataops.grid_ops as gridOps
lfi = gridOps.shuffle_feature_IDs(lfi)

## View again after shuffling

Same view as before, now with the shuffled IDs. Also wrapped in quotes, so take the quotes off to run it.

In [ ]:
"""from upxo.viz import gsviz

gsviz.plot_pvgrid(gsviz.make_pvgrid(lfi, scalar_name='my_day'),
                  scalar_name='my_day', cmap='nipy_spectral', show_edges=True)"""